In [1]:
#%%
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path

from monoculture.analysis.setup import (
    ACS_TASKS,
    TABLESHIFT_TASKS,
    LLM_MODELS,
)
from monoculture.analysis.utils import (
    model_to_key,
    create_result_df,
    key_to_model
)
import pandas as pd

RESULTS_ROOT_DIR = Path("./results/")

### Check availability for same prompting style

In [3]:
TASKS = ACS_TASKS + TABLESHIFT_TASKS

subfolders = [
    "tableshift/0-bullet-is",
    "tableshift/10-reuse-bullet-is",
    "folktexts/0-bullet-is",
    "folktexts/10-reuse-bullet-is",
    # "folktexts/0-bullet-colon",
    # "folktexts/0-bullet-equal",
    # "folktexts/0-text-is",
]  # , 'few-shot']
TASKS

('ACSIncome',
 'ACSEmployment',
 'ACSTravelTime',
 'ACSPublicCoverage',
 'BRFSS_Blood_Pressure')

In [4]:
SAVE_DIR = RESULTS_ROOT_DIR
save_file_path = SAVE_DIR / "overview_results_by_prompt_style.csv"

In [5]:
load_df = False
df = (
    pd.read_csv(save_file_path)
    if load_df
    else create_result_df(
        RESULTS_ROOT_DIR, subfolders=subfolders, tasks=TASKS, #save_path=save_file_path
    )
)

Shape of df:  (721, 14)


In [6]:
print(df.shape)
df[
    (df["num_shots"] == 0) & (df["task"] == "ACSIncome") & (df["threshold_fitted"] == 0)
].head()

(721, 14)


,task,model,is_inst,threshold_fitted,threshold,accuracy,bench_hash,num_shots,prompt_format,prompt_connector,prompt_granularity,prompt_feature_order,eval_results_path,predictions_path
0,ACSIncome,meta-llama--Meta-Llama-3.2-1B-Instruct,1,0,0.5,0.367876,4087643733,0,bullet,is,original,default,results/folktexts/0-bullet-is/model-meta-llama...,results/folktexts/0-bullet-is/model-meta-llama...
3,ACSIncome,allenai--OLMo-2-1124-7B-Instruct,1,0,0.5,0.404728,2229201787,0,bullet,is,original,default,results/folktexts/0-bullet-is/model-allenai--O...,results/folktexts/0-bullet-is/model-allenai--O...
5,ACSIncome,google--gemma-2-27b,0,0,0.5,0.523749,1719521954,0,bullet,is,original,default,results/folktexts/0-bullet-is/model-google--ge...,results/folktexts/0-bullet-is/model-google--ge...
7,ACSIncome,google--gemma-2-27b-it,1,0,0.5,0.590291,1665971878,0,bullet,is,original,default,results/folktexts/0-bullet-is/model-google--ge...,results/folktexts/0-bullet-is/model-google--ge...
8,ACSIncome,meta-llama--Meta-Llama-3-70B,0,0,0.5,0.669757,758164074,0,bullet,is,original,default,results/folktexts/0-bullet-is/model-meta-llama...,results/folktexts/0-bullet-is/model-meta-llama...


In [7]:
df['task'].unique()

array(['ACSIncome', 'ACSEmployment', 'ACSTravelTime', 'ACSPublicCoverage',
       'BRFSS_Blood_Pressure'], dtype=object)

In [9]:
show_available = True
show_unavailable = True
fitted_treshold = True

for task_name in TASKS: #ACS_TASKS[:2] + TABLESHIFT_TASKS:
    print(task_name)
    for num_shots in [0,10]:
        for format in ["bullet"]:  # ['bullet', 'text']:
            for con in ["is"]:  # ['is', 'colon', 'equal']:
                if format == "text" and con != "is":
                    continue
                else:
                    print(f"  {num_shots} {format} {con}")
                model_str = ""
                for m in LLM_MODELS:
                    num_entries = df[
                        (df["task"] == task_name)
                        & (df["model"] == model_to_key(m))
                        & (df["prompt_format"] == format)
                        & (df["prompt_connector"] == con)
                        & (df["num_shots"] == num_shots)
                        & (df["threshold_fitted"] == int(fitted_treshold))
                    ].shape[0]
                    if show_available:
                        if num_entries == 1:
                            pass
                            # print(f"\t- {m} ")
                            # print(f"--model={m}", end=" ")
                        elif num_entries > 1:
                            print(
                                f"\t- {m} -- Found multiple models with given characteristics."
                            )
                    if show_unavailable and num_entries == 0:
                        print(f"\tx {m}")  # , end =" ")
                        model_str+=f"--model={m} "
                        # print(f"--model={m}", end =" ")
                if len(model_str)>0:
                    print(f"--task={task_name} {model_str}", end=" ")
                print()

ACSIncome
  0 bullet is

  10 bullet is

ACSEmployment
  0 bullet is

  10 bullet is

ACSTravelTime
  0 bullet is

  10 bullet is
	x google/gemma-1.1-2b-it
	x google/gemma-2-9b
	x google/gemma-2-9b-it
	x meta-llama/Meta-Llama-3-70B
	x meta-llama/Meta-Llama-3-70B-Instruct
	x meta-llama/Meta-Llama-3.1-8B-Instruct
	x meta-llama/Meta-Llama-3.1-70B
	x meta-llama/Meta-Llama-3.1-70B-Instruct
	x meta-llama/Meta-Llama-3.3-70B-Instruct
	x mistralai/Mixtral-8x7B-v0.1
	x mistralai/Mixtral-8x7B-Instruct-v0.1
	x mistralai/Mixtral-8x22B-v0.1
	x mistralai/Mixtral-8x22B-Instruct-v0.1
	x mistralai/Mistral-Small-24B-Base-2501
	x mistralai/Mistral-Small-24B-Instruct-2501
	x 01-ai/Yi-34B
	x 01-ai/Yi-34B-Chat
	x Qwen/Qwen2-7B
	x Qwen/Qwen2-7B-Instruct
	x Qwen/Qwen2-72B
	x Qwen/Qwen2-72B-Instruct
	x Qwen/Qwen2.5-7B
	x Qwen/Qwen2.5-7B-Instruct
	x Qwen/Qwen2.5-72B
	x Qwen/Qwen2.5-72B-Instruct
	x allenai/OLMo-7B-Instruct-hf
--task=ACSTravelTime --model=google/gemma-1.1-2b-it --model=google/gemma-2-9b --model=go

### Check availability for different prompting style

In [72]:
from monoculture.analysis.setup import variations, map_short_to_feature_order
variations

{'feature_order': ['default', 'rand 1', 'rand 2', 'rand 3', ' reversed'],
 'format': ['bullet', 'text', 'comma'],
 'connector': ['is', '=', ':'],
 'granularity': ['original', 'low']}

In [73]:
TASKS = ['ACSIncome']
MODELS = ['Qwen--Qwen2.5-7B-Instruct',
  'Qwen--Qwen2.5-72B-Instruct',
  'meta-llama--Meta-Llama-3-8B-Instruct',
  'meta-llama--Meta-Llama-3.3-70B-Instruct']
subfolders = [
    "folktexts/0-variations",
    "folktexts/10-variations",
]
SAVE_DIR = RESULTS_ROOT_DIR
save_file_path = SAVE_DIR / "overview_results_variations.csv"

In [74]:
load_df = False
df = (
    pd.read_csv(save_file_path)
    if load_df
    else create_result_df(
        RESULTS_ROOT_DIR, subfolders=subfolders, tasks=TASKS, #save_path=save_file_path
    )
)

Shape of df:  (1154, 14)


In [75]:
print(df.shape)
df[
    (df["num_shots"] == 0) & (df["task"] == "ACSIncome") & (df["threshold_fitted"] == 1)
].head()

(1154, 14)


,task,model,is_inst,threshold_fitted,threshold,accuracy,bench_hash,num_shots,prompt_format,prompt_connector,prompt_granularity,prompt_feature_order,eval_results_path,predictions_path
0,ACSIncome,Qwen--Qwen2.5-72B-Instruct,1,1,0.164835,0.761706,175002779,0,text,=,low,reversed,results/folktexts/0-variations/model-Qwen--Qwe...,results/folktexts/0-variations/model-Qwen--Qwe...
1,ACSIncome,Qwen--Qwen2.5-72B-Instruct,1,1,0.148014,0.776888,1344084919,0,bullet,is,original,reversed,results/folktexts/0-variations/model-Qwen--Qwe...,results/folktexts/0-variations/model-Qwen--Qwe...
2,ACSIncome,Qwen--Qwen2.5-72B-Instruct,1,1,0.003172,0.766957,3156286684,0,text,is,original,default,results/folktexts/0-variations/model-Qwen--Qwe...,results/folktexts/0-variations/model-Qwen--Qwe...
3,ACSIncome,Qwen--Qwen2.5-72B-Instruct,1,1,0.003183,0.772761,2985942201,0,bullet,:,original,default,results/folktexts/0-variations/model-Qwen--Qwe...,results/folktexts/0-variations/model-Qwen--Qwe...
4,ACSIncome,Qwen--Qwen2.5-72B-Instruct,1,1,0.007574,0.779946,1338756957,0,bullet,is,original,default,results/folktexts/0-variations/model-Qwen--Qwe...,results/folktexts/0-variations/model-Qwen--Qwe...


In [76]:
import os
from datetime import datetime

In [77]:
show_available = False
show_unavailable = True
fitted_treshold = True

for task_name in TASKS:  # ACS_TASKS[:2] + TABLESHIFT_TASKS:
    print(task_name)
    for m in MODELS:
        # print(m)
        for num_shots in [0]:
            for format in variations["format"]:
                for con in variations["connector"]:
                    for order in variations["feature_order"]:
                        for gran in variations["granularity"]:
                            model_str = ""
                            df_variation = df[
                                (df["task"] == task_name)
                                & (df["model"] == model_to_key(m))
                                & (df["prompt_format"] == format)
                                & (df["prompt_connector"] == con)
                                & (df["prompt_feature_order"] == order)
                                & (df["prompt_granularity"] == gran)
                                & (df["num_shots"] == num_shots)
                                & (df["threshold_fitted"] == int(fitted_treshold))
                            ]
                            num_entries = df_variation.shape[0]
                            if show_available:
                                if num_entries == 1:
                                    pass
                                    # print(f"\t- {num_shots} {format} {con} {order} {gran}")
                                    # print(f"--model={m}", end=" ")
                                elif num_entries > 1:
                                    print(
                                        f"\t- {num_shots} {format} {con} {order} {gran} -- Found multiple models with given characteristics."
                                    )

                                    for file in df_variation[
                                        "eval_results_path"
                                    ].values:
                                        mod_time = os.path.getmtime(file)
                                        mod_datetime = datetime.fromtimestamp(mod_time)
                                        print(Path(file).parent, mod_datetime)
                                    newest = max(
                                        df_variation["eval_results_path"].values,
                                        key=os.path.getmtime,
                                    )
                                    for file in df_variation[
                                        "eval_results_path"
                                    ].values:
                                        if file != newest:
                                            print(f"remove {Path(file).as_posix()}")
                                            # os.remove(Path(file).as_posix())
                            if show_unavailable and num_entries == 0:
                                #print(f"\n\tx {m}")  # , end =" ")
                                # print(f"{m} --variation=\"format={format};connector={con};granularity={gran};order={map_short_to_feature_order[order]}")
                                model_str += f"--model={key_to_model(m)} \n"
                                # print(f"--model={m}", end =" ")
                            if len(model_str) > 0:
                                print(
                                    f"python -m folktexts.cli.launch_experiments_htcondor --executable-path ./folktexts/cli/run_benchmark.py --results-dir '/fast/mgorecki/monoculture/results/folktexts/{num_shots}-variations' --logger-level=INFO --models-dir /fast/mgorecki/models/ --task=ACSIncome --variation=\"format={format};connector={con};granularity={gran};order={map_short_to_feature_order[order]}\" {model_str}"
                                    + (" --reuse-few-shot-examples=True --few-shot=10 --balance-few-shot-examples=True"
                                    if num_shots == 10
                                    else "")
                                )  # {model_str}")
        # print('\n')

ACSIncome
python -m folktexts.cli.launch_experiments_htcondor --executable-path ./folktexts/cli/run_benchmark.py --results-dir '/fast/mgorecki/monoculture/results/folktexts/0-variations' --logger-level=INFO --models-dir /fast/mgorecki/models/ --task=ACSIncome --variation="format=text;connector==;granularity=original;order=AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P" --model=Qwen/Qwen2.5-7B-Instruct 

python -m folktexts.cli.launch_experiments_htcondor --executable-path ./folktexts/cli/run_benchmark.py --results-dir '/fast/mgorecki/monoculture/results/folktexts/0-variations' --logger-level=INFO --models-dir /fast/mgorecki/models/ --task=ACSIncome --variation="format=comma;connector==;granularity=original;order=AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P" --model=Qwen/Qwen2.5-7B-Instruct 

python -m folktexts.cli.launch_experiments_htcondor --executable-path ./folktexts/cli/run_benchmark.py --results-dir '/fast/mgorecki/monoculture/results/folktexts/0-variations' --logger-level=I